In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install anndata scanpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 84.4 MB/s eta 0:00:00


In [ ]:
import anndata as ad

path = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"
adata = ad.read_h5ad(path, backed='r')

In [ ]:
print(adata.obs.columns)

Index(['sampleID', 'donor_id', 'CoVID-19 severity', 'celltype', 'majorType',
       'City', '_src_row'],
      dtype='object')


In [ ]:
import anndata as ad
import pandas as pd

# ---- inputs ----
ADATA_PATH = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"

COL_SAMPLE = "sampleID"
COL_DONOR  = "donor_id"
COL_Y      = "CoVID-19 severity"   # categories: control / mild/moderate / severe/critical

# ---- load ----
adata = ad.read_h5ad(ADATA_PATH)

# ---- sanity: required columns exist ----
missing = [c for c in [COL_SAMPLE, COL_DONOR, COL_Y] if c not in adata.obs.columns]
assert not missing, f"Missing obs columns: {missing}. Available: {list(adata.obs.columns)}"

obs = adata.obs[[COL_SAMPLE, COL_DONOR, COL_Y]].copy()
obs[COL_SAMPLE] = obs[COL_SAMPLE].astype(str)
obs[COL_DONOR]  = obs[COL_DONOR].astype(str)
obs[COL_Y]      = obs[COL_Y].astype(str)

print("=== Cell-level overview ===")
print("n_cells:", adata.n_obs)
print("n_genes:", adata.n_vars)
print("n_samples (unique sampleID):", obs[COL_SAMPLE].nunique())
print("n_donors  (unique donor_id):", obs[COL_DONOR].nunique())
print()

# ---- build per-sample table ----
g = obs.groupby(COL_SAMPLE, sort=False)

sample_tbl = pd.DataFrame({
    "n_cells": g.size(),
    "n_donors_in_sample": g[COL_DONOR].nunique(),
    "n_severity_in_sample": g[COL_Y].nunique(),
})
# take representative (only valid if uniq==1; otherwise still shows one example)
sample_tbl["donor_id_rep"] = g[COL_DONOR].first()
sample_tbl["severity_rep"] = g[COL_Y].first()

bad_donor = sample_tbl.query("n_donors_in_sample != 1")
bad_y     = sample_tbl.query("n_severity_in_sample != 1")

print("=== Integrity checks (sample-level) ===")
print(f"[check] sampleID -> donor_id single-valued: {'OK' if len(bad_donor)==0 else 'FAIL'} "
      f"(bad samples: {len(bad_donor)})")
if len(bad_donor) > 0:
    print("Top offending sampleIDs (showing up to 20):")
    print(bad_donor.sort_values(["n_donors_in_sample", "n_cells"], ascending=False).head(20))
print()

print(f"[check] sampleID -> severity single-valued: {'OK' if len(bad_y)==0 else 'FAIL'} "
      f"(bad samples: {len(bad_y)})")
if len(bad_y) > 0:
    print("Top offending sampleIDs (showing up to 20):")
    print(bad_y.sort_values(["n_severity_in_sample", "n_cells"], ascending=False).head(20))
print()

# ---- class ratios (bag-level and cell-level) ----
# keep only "clean" samples for meaningful ratios
clean_samples = sample_tbl.query("n_donors_in_sample == 1 and n_severity_in_sample == 1").copy()

print("=== Class distribution (bag-level: by sampleID) ===")
bag_counts = clean_samples["severity_rep"].value_counts(dropna=False).sort_index()
bag_props = (bag_counts / bag_counts.sum()).sort_values(ascending=False)
print(pd.DataFrame({"n_samples": bag_counts, "prop": bag_props}))
print()

print("=== Class distribution (cell-level: by cells) ===")
# If there are dirty samples, this still counts per-cell as-is; that's fine as a descriptive stat.
cell_counts = obs[COL_Y].value_counts(dropna=False).sort_index()
cell_props = (cell_counts / cell_counts.sum()).sort_values(ascending=False)
print(pd.DataFrame({"n_cells": cell_counts, "prop": cell_props}))
print()

# Optional: quick summary of donor multiplicity per class (bag-level)
print("=== Donor counts per class (bag-level, clean samples) ===")
donor_per_class = clean_samples.groupby("severity_rep")["donor_id_rep"].nunique().sort_values(ascending=False)
print(donor_per_class)


=== Cell-level overview ===
n_cells: 914746
n_genes: 4000
n_samples (unique sampleID): 282
n_donors  (unique donor_id): 196

=== Integrity checks (sample-level) ===
[check] sampleID -> donor_id single-valued: OK (bad samples: 0)

[check] sampleID -> severity single-valued: OK (bad samples: 0)

=== Class distribution (bag-level: by sampleID) ===
                 n_samples      prop
severity_rep                        
control                 28  0.099291
mild/moderate          122  0.432624
severe/critical        132  0.468085

=== Class distribution (cell-level: by cells) ===
                   n_cells      prop
CoVID-19 severity                   
control             108272  0.118363
mild/moderate       431275  0.471470
severe/critical     375199  0.410167

=== Donor counts per class (bag-level, clean samples) ===
severity_rep
severe/critical    92
mild/moderate      79
control            25
Name: donor_id_rep, dtype: int64


In [ ]:
import h5py
import numpy as np

PATHS = {
    "ORG": "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad",
    "RAW": "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts.h5ad",
    "HVG": "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad",
}

def _decode(x):
    if isinstance(x, (bytes, np.bytes_)):
        return x.decode("utf-8", errors="replace")
    return str(x)

def inspect_obs_minimal(path, cols=("sampleID","donor_id","CoVID-19 severity"), n=20):
    print("\n===", path, "===")
    with h5py.File(path, "r") as f:
        # common layouts vary by anndata version; list obs group keys
        if "obs" not in f:
            print("[NO /obs group] keys:", list(f.keys())[:20])
            return
        obs = f["obs"]
        print("[/obs keys sample]", list(obs.keys())[:30])

        # case 1: old-style dataframe encoding: datasets directly under /obs
        for c in cols:
            if c in obs:
                ds = obs[c]
                print(f"\n[obs/{c}] type={type(ds)}")
                if isinstance(ds, h5py.Dataset):
                    print("  dtype:", ds.dtype, "shape:", ds.shape)
                    head = [_decode(x) for x in ds[:min(n, ds.shape[0])]]
                    print("  head:", head[:min(10, len(head))])
                else:
                    print("  (not a dataset) keys:", list(ds.keys())[:20])

        # case 2: newer encoding: /obs is a group with "_index" + "columns"/"data"
        # try to detect and print any fixed-length string dtypes that look like truncation.
        print("\n[scan for suspicious fixed-length strings under /obs]")
        for k in list(obs.keys())[:200]:
            obj = obs[k]
            if isinstance(obj, h5py.Dataset) and obj.dtype.kind == "S":
                print("  ", k, "dtype", obj.dtype, "example", _decode(obj[0]))

for tag, p in PATHS.items():
    print("\n###", tag)
    inspect_obs_minimal(p)



### ORG

=== /content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad ===
[/obs keys sample] ['BCR single cell sequencing', 'COVID-19-related medication and anti-microbials', 'City', 'CoVID-19 severity', 'Comorbidities', 'Leukocytes [G over L]', 'Lymphocytes [G over L]', 'Neutrophils [G over L]', 'Outcome', 'Sample time', 'Sample type', 'Sampling day (Days after symptom onset)', 'TCR single cell sequencing', 'Unpublished', '_index', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'celltype', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'majorType', 'observation_joinid', 'sampleID', 'self_reported_ethnicity']

[obs/sampleID] type=<class 'h5py._hl.group.Group'>
  (not a dataset) keys: ['categories', 'codes']

[obs/donor_id] type=<class 'h5py._hl.group.Group'>
  (not a dataset) keys: ['categories', 'codes']

[obs/CoVID-19 severity] type=<class 'h5p

In [ ]:
import h5py
import numpy as np
import os

# === 설정: 파일 경로 ===
PATH_ORG = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad"

# 복구할 파일들의 리스트
TARGET_FILES = [
    # 1. HVG Subset (이미 복구되었어도 안전하게 재확인)
    "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad",
    # 2. Raw Counts (아까 에러나서 못 한 것)
    "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts.h5ad"
]

COLS_TO_FIX = ["sampleID", "donor_id", "CoVID-19 severity"]

# === 유틸리티 함수 ===
def _as_str(arr):
    """바이트 문자열 등을 유니코드로 변환"""
    arr = np.asarray(arr)
    if arr.dtype.kind == "U":
        return arr
    if arr.dtype.kind in ("S", "O"):
        return arr.astype("U")
    return arr.astype("U")

def decode_cat_group_fast(source_group, row_indexer):
    """
    구글 드라이브 최적화 버전:
    데이터를 조금씩 읽지 않고(Random Read), 한 번에 메모리에 올린 후(Bulk Read) 처리합니다.
    """
    row_indexer = row_indexer.astype(np.int64)
    n_target = len(row_indexer)

    # 1. [Speed Up] 원본의 코드와 카테고리를 통째로 RAM에 적재 (수 MB 수준이라 빠르고 안전함)
    all_codes = source_group["codes"][()]
    cats = _as_str(source_group["categories"][()])

    # 2. 인덱싱 준비
    # (메모리 상 작업이므로 argsort가 필수는 아니지만, 로직 안정성을 위해 유지)
    perm = np.argsort(row_indexer)
    sorted_rows = row_indexer[perm]

    # 3. [In-Memory Slicing] RAM에서 즉시 슬라이싱
    codes_sorted = all_codes[sorted_rows].astype(np.int64)

    # 4. 카테고리 매핑 (Vectorized lookup이 가능하지만, 안전하게 루프 처리)
    out_sorted = np.empty(n_target, dtype=object)

    # -1(결측치) 처리 포함
    for i, c in enumerate(codes_sorted):
        out_sorted[i] = "" if c < 0 else str(cats[int(c)])

    # 5. 원래 순서로 복원 (Inverse Permutation)
    inv = np.empty_like(perm)
    inv[perm] = np.arange(n_target)
    out = out_sorted[inv]

    return out

def run_recovery_for_file(target_path):
    """
    단일 파일에 대한 복구를 수행하는 함수.
    변수 스코프를 격리하여 '닫힌 파일 참조' 오류를 방지합니다.
    """
    file_name = os.path.basename(target_path)
    print(f"\n>>> Processing: {file_name}")

    if not os.path.exists(target_path):
        print(f"    [Error] File not found: {target_path}")
        return

    # r+ 모드: 읽기 및 수정
    with h5py.File(PATH_ORG, "r") as f_org, h5py.File(target_path, "r+") as f_target:
        obs_org = f_org["obs"]
        obs_target = f_target["obs"]

        # 1. _src_row 확인
        if "_src_row" not in obs_target:
            print("    [Fail] '_src_row' column is missing. Cannot recover.")
            return

        # 전체 인덱스 로딩 (Bulk Load)
        src_indices = obs_target["_src_row"][()].astype(np.int64)
        n_cells = len(src_indices)
        print(f"    Target Cells: {n_cells}, Source Index Range: {src_indices.min()} ~ {src_indices.max()}")
        print(f"    Unique Source Rows: {len(np.unique(src_indices))} (Should be close to n_cells)")

        str_dt = h5py.string_dtype("utf-8")

        # 2. 컬럼별 복구 수행
        for col in COLS_TO_FIX:
            if col not in obs_org:
                print(f"    [Skip] Column '{col}' not found in Source file.")
                continue

            print(f"    Recovering '{col}'...", end=" ", flush=True)

            # 고속 디코딩 함수 호출
            decoded_data = decode_cat_group_fast(obs_org[col], src_indices)

            # 기존 데이터셋 삭제 후 재생성
            if col in obs_target:
                del obs_target[col]

            obs_target.create_dataset(col, data=decoded_data, dtype=str_dt)

            # 검증용 샘플 출력 (다양성을 확인하기 위해 등간격 샘플링)
            # 0%, 25%, 50%, 75%, 100% 지점의 데이터를 찍어봄
            chk_idx = np.linspace(0, n_cells - 1, 5).astype(int)
            samples = decoded_data[chk_idx]

            print("Done.")
            print(f"      Samples (indices {chk_idx}): {samples}")

    print(f"<<< Completed: {file_name}")

# === 메인 실행 ===
if __name__ == "__main__":
    print("=== MULTI-FILE METADATA RECOVERY START ===")

    for fpath in TARGET_FILES:
        run_recovery_for_file(fpath)

    print("\n=== ALL TASKS FINISHED ===")

=== MULTI-FILE METADATA RECOVERY START ===

>>> Processing: adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad
    Target Cells: 914746, Source Index Range: 0 ~ 1462698
    Unique Source Rows: 914746 (Should be close to n_cells)
    Recovering 'sampleID'... Done.
      Samples (indices [     0 228686 457372 686058 914745]): ['S-S070-1' 'S-M062-2' 'S-M004-6' 'S-M031-2' 'S-S053']
    Recovering 'donor_id'... Done.
      Samples (indices [     0 228686 457372 686058 914745]): ['P-S070' 'P-M062' 'P-M004' 'P-M031' 'P-S053']
    Recovering 'CoVID-19 severity'... Done.
      Samples (indices [     0 228686 457372 686058 914745]): ['severe/critical' 'mild/moderate' 'mild/moderate' 'mild/moderate'
 'severe/critical']
<<< Completed: adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad

>>> Processing: adata_cap4096_min100_milmin_rawcounts.h5ad
    Target Cells: 914746, Source Index Range: 0 ~ 1462698
    Unique Source Rows: 914746 (Should be close to n_cells)
    Recovering 'sampleID'... Done.
   

In [ ]:
!pip install anndata scanpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 95.8 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
import h5py
import anndata as ad
import scipy.sparse as sp

PATH = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"

COL_SAMPLE = "sampleID"
COL_DONOR  = "donor_id"
COL_Y      = "CoVID-19 severity"
COL_SRCROW = "_src_row"

EXPECTED_Y = {"control", "mild/moderate", "severe/critical"}

def _assert(cond, msg):
    if not cond:
        raise AssertionError(msg)

def check_h5_structure(path, seed=0):
    print("\n[H5] structure/spec checks", flush=True)
    rng = np.random.default_rng(seed)

    with h5py.File(path, "r") as f:
        _assert("X" in f, "missing /X")
        _assert("obs" in f, "missing /obs")
        _assert("var" in f, "missing /var")

        X = f["X"]
        _assert(isinstance(X, h5py.Group), "/X is not a group (expected CSR group)")
        for k in ["data", "indices", "indptr"]:
            _assert(k in X, f"missing /X/{k} (CSR)")

        data = X["data"]
        indices = X["indices"]
        indptr = X["indptr"]

        _assert(data.ndim == 1 and indices.ndim == 1 and indptr.ndim == 1, "CSR arrays must be 1D")
        _assert(indptr.shape[0] >= 2, "indptr too short")
        _assert(indptr[0] == 0, "CSR indptr[0] must be 0")

        # [OPTIMIZATION] Check last element without reading full array
        _assert(indptr[-1] == data.shape[0], "CSR indptr[-1] must equal nnz")
        _assert(indices.shape[0] == data.shape[0], "CSR indices/data length mismatch")

        if "shape" in X.attrs:
            sh = tuple(X.attrs["shape"])
            _assert(len(sh) == 2, "X.attrs['shape'] must be length-2")
        else:
            print("  [warn] /X has no shape attr (not always present)", flush=True)

        # [FIX: CRITICAL PERFORMANCE] Use Contiguous Slicing instead of Random Access
        nnz = indices.shape[0]
        n_check = min(200000, nnz)
        if n_check > 0:
            # 랜덤한 '위치' 하나를 잡아서 거기서부터 연속으로 읽음 (Single IO Request)
            start_idx = rng.integers(0, max(1, nnz - n_check))
            end_idx = min(nnz, start_idx + n_check)

            idx_vals = indices[start_idx : end_idx]
            _assert(np.all(idx_vals >= 0), "CSR indices contain negative")

            if "_index" in f["var"]:
                n_vars = f["var"]["_index"].shape[0]
                _assert(np.all(idx_vals < n_vars), "CSR indices exceed n_vars")

        obs = f["obs"]
        for c in [COL_SAMPLE, COL_DONOR, COL_Y]:
            _assert(c in obs, f"missing obs/{c}")
            ds = obs[c]
            # [NOTE] h5py Dataset checks are fast enough
            _assert(isinstance(ds, h5py.Dataset), f"obs/{c} is not dataset")
            _assert(ds.dtype.kind in ("S", "O"), f"obs/{c} dtype unexpected: {ds.dtype}")

        if COL_SRCROW in obs:
            ds = obs[COL_SRCROW]
            _assert(isinstance(ds, h5py.Dataset), "obs/_src_row must be dataset")
            _assert(ds.dtype.kind in ("i", "u"), "obs/_src_row should be integer dtype")

    print("  [OK] H5 structure/spec looks consistent", flush=True)

def check_semantics_with_anndata(path):
    print("\n[AnnData] semantic checks", flush=True)

    # [FIX: MEMORY] backed='r' ensures fast load
    adata = ad.read_h5ad(path, backed="r")

    _assert(adata.n_obs > 1000, "n_obs suspiciously small")
    _assert(adata.n_vars == 4000, f"n_vars expected 4000, got {adata.n_vars}")

    # [FIX: BACKED COMPATIBILITY] Backed SparseMatrix is not technically scipy.sparse
    # Check if it has sparse attributes or is explicitly sparse
    is_sparse = sp.issparse(adata.X) or (hasattr(adata.X, "format") and adata.X.format == "csr")
    _assert(is_sparse, "adata.X is not sparse (or not recognized as CSR)")
    _assert(adata.X.shape == (adata.n_obs, adata.n_vars), "X shape mismatch")

    # [FIX: FAST VALUE CHECK] Load only a tiny chunk into memory to check values
    # Reading random indices from backed X is slow. Reading a small contiguous block is fast.
    n_sample_cells = min(500, adata.n_obs)
    X_chunk = adata[:n_sample_cells].X

    # If backed, slicing returns a sparse matrix in memory.
    if sp.issparse(X_chunk):
        vals = X_chunk.data
    else:
        vals = X_chunk.flatten()
        vals = vals[vals != 0] # simulate sparse data check

    if len(vals) > 0:
        _assert(vals.min() >= 0, "X has negative values")
        _assert(np.all(np.isfinite(vals)), "X has non-finite values")
        _assert(np.allclose(vals, np.round(vals)), "X values not integer-like")

    # Metadata checks (obs is already loaded in memory even in backed mode usually, or loaded on demand)
    for c in [COL_SAMPLE, COL_DONOR, COL_Y]:
        _assert(c in adata.obs.columns, f"missing obs col {c}")

    n_samples = adata.obs[COL_SAMPLE].nunique()
    n_donors  = adata.obs[COL_DONOR].nunique()
    _assert(n_samples > 10, f"sampleID nunique too small: {n_samples}")
    _assert(n_donors  > 10, f"donor_id nunique too small: {n_donors}")

    yvals = set(map(str, adata.obs[COL_Y].unique()))
    _assert(yvals.issubset(EXPECTED_Y), f"unexpected severity values: {sorted(yvals)}")
    _assert(len(yvals) == 3, f"severity does not have 3 classes: {sorted(yvals)}")

    g = adata.obs.groupby(COL_SAMPLE, sort=False)
    bad_donor = g[COL_DONOR].nunique()
    bad_y     = g[COL_Y].nunique()
    _assert((bad_donor == 1).all(), f"sampleID->donor_id not single-valued: {(bad_donor!=1).sum()} bad samples")
    _assert((bad_y == 1).all(), f"sampleID->severity not single-valued: {(bad_y!=1).sum()} bad samples")

    sample_tbl = g[[COL_DONOR, COL_Y]].first()
    print("  n_cells:", adata.n_obs, "n_genes:", adata.n_vars, flush=True)
    print("  n_samples:", n_samples, "n_donors:", n_donors, flush=True)
    print("\n  [bag-level class counts]", flush=True)
    print(sample_tbl[COL_Y].value_counts(), flush=True)

    print("\n  [OK] AnnData semantic checks passed", flush=True)

def run_all(path):
    print("=== VALIDATION START ===", flush=True)
    print("file:", path, flush=True)

    if not os.path.exists(path):
         raise FileNotFoundError(f"File not found: {path}")

    print("size (GB):", round(os.path.getsize(path) / (1024**3), 3), flush=True)

    check_h5_structure(path)
    check_semantics_with_anndata(path)
    print("\n=== VALIDATION DONE: PASS ===", flush=True)

import shutil

# 1. 파일 복사 (Google Drive -> Colab Local VM)
LOCAL_PATH = "/content/temp_data.h5ad"

if not os.path.exists(LOCAL_PATH):
    print(f"Copying file to local storage... (Wait a moment)", flush=True)
    shutil.copy(PATH, LOCAL_PATH)
    print("Copy complete.", flush=True)

# 2. 로컬 경로로 검증 실행
run_all(LOCAL_PATH)

Copying file to local storage... (Wait a moment)
Copy complete.
=== VALIDATION START ===
file: /content/temp_data.h5ad
size (GB): 2.351

[H5] structure/spec checks
  [OK] H5 structure/spec looks consistent

[AnnData] semantic checks
  n_cells: 914746 n_genes: 4000
  n_samples: 282 n_donors: 196

  [bag-level class counts]
CoVID-19 severity
severe/critical    132
mild/moderate      122
control             28
Name: count, dtype: int64

  [OK] AnnData semantic checks passed

=== VALIDATION DONE: PASS ===


In [ ]:
import anndata as ad
import pandas as pd

COL_SAMPLE = "sampleID"
COL_DONOR  = "donor_id"
COL_Y      = "CoVID-19 severity"

# 구글 드라이브 경로 대신, 아까 복사한 로컬 경로 사용 (속도 10배 이상 향상)
adata = ad.read_h5ad("/content/temp_data.h5ad", backed="r")
obs = adata.obs[[COL_SAMPLE, COL_DONOR, COL_Y]]

print("=== HVG4000 obs nunique checks ===")
print("n_cells:", adata.n_obs, "n_genes:", adata.n_vars)
print("nunique(sampleID):", obs[COL_SAMPLE].nunique())
print("nunique(donor_id):",  obs[COL_DONOR].nunique())
print("nunique(severity):",  obs[COL_Y].nunique())

print("\n=== examples (head) ===")
print(obs.head(10))

print("\n=== severity value counts (cell-level) ===")
print(obs[COL_Y].value_counts())

# bag-level class counts: sampleID -> severity should be single-valued
g = obs.groupby(COL_SAMPLE, sort=False)[COL_Y]
n_y_per_sample = g.nunique()
bad = n_y_per_sample[n_y_per_sample != 1]

print("\n=== sampleID -> severity single-valued check ===")
if len(bad) == 0:
    print("[OK] all sampleIDs map to exactly 1 severity")
else:
    print(f"[FAIL] {len(bad)} sampleIDs map to !=1 severities")
    print(bad.sort_values(ascending=False).head(20))

# bag-level donor consistency: sampleID -> donor_id should be single-valued
g2 = obs.groupby(COL_SAMPLE, sort=False)[COL_DONOR]
n_d_per_sample = g2.nunique()
bad2 = n_d_per_sample[n_d_per_sample != 1]

print("\n=== sampleID -> donor_id single-valued check ===")
if len(bad2) == 0:
    print("[OK] all sampleIDs map to exactly 1 donor_id")
else:
    print(f"[FAIL] {len(bad2)} sampleIDs map to !=1 donor_ids")
    print(bad2.sort_values(ascending=False).head(20))

# bag-level class counts (by sample)
if len(bad) == 0:
    sample_y = obs.groupby(COL_SAMPLE, sort=False)[COL_Y].first()
    print("\n=== bag-level class counts (by sampleID) ===")
    print(sample_y.value_counts())


=== HVG4000 obs nunique checks ===
n_cells: 914746 n_genes: 4000
nunique(sampleID): 282
nunique(donor_id): 196
nunique(severity): 3

=== examples (head) ===
   sampleID donor_id CoVID-19 severity
0  S-S070-1   P-S070   severe/critical
1  S-S070-1   P-S070   severe/critical
2  S-S070-1   P-S070   severe/critical
3  S-S070-1   P-S070   severe/critical
4  S-S070-1   P-S070   severe/critical
5  S-S070-1   P-S070   severe/critical
6  S-S070-1   P-S070   severe/critical
7  S-S070-1   P-S070   severe/critical
8  S-S070-1   P-S070   severe/critical
9  S-S070-1   P-S070   severe/critical

=== severity value counts (cell-level) ===
CoVID-19 severity
mild/moderate      431275
severe/critical    375199
control            108272
Name: count, dtype: int64

=== sampleID -> severity single-valued check ===
[OK] all sampleIDs map to exactly 1 severity

=== sampleID -> donor_id single-valued check ===
[OK] all sampleIDs map to exactly 1 donor_id

=== bag-level class counts (by sampleID) ===
CoVID-19 sev

In [13]:
import os, json, time, shutil, hashlib
import numpy as np
import pandas as pd
import anndata as ad

# ============================================================
# Balanced Donor Assignment Splitter (A 방식: 큰 donor 분산)
# - donor 단위로 split (donor-holdout)
# - 초기 배치: bagcount 큰 donor부터 "부족한 split"에 분산
# - 보정: donor 이동/스왑으로 bag target(197/28/57) 맞춤
# - 제약: class minima + val/test donor minima + (선택) prior shift 완화
# ============================================================

# ------------------------
# 1) Config
# ------------------------
DRIVE_PATH = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"
LOCAL_PATH = "/content/temp_data_hvg4000.h5ad"
OUT_ROOT   = "/content/drive/MyDrive/scMILD_residual/splits_donor_holdout"

COL_SAMPLE = "sampleID"
COL_DONOR  = "donor_id"
COL_Y      = "CoVID-19 severity"

R_TRAIN, R_VAL, R_TEST = 0.70, 0.10, 0.20

# Hard minima (bags)
MIN_PER_CLASS_TRAIN = {"control": 15, "mild/moderate": 50, "severe/critical": 50}
MIN_PER_CLASS_VAL   = {"control": 4,  "mild/moderate": 8,  "severe/critical": 8}
MIN_PER_CLASS_TEST  = {"control": 6,  "mild/moderate": 10, "severe/critical": 10}

# Hard minima (donors)
MIN_DONORS_VAL  = 20
MIN_DONORS_TEST = 20

# Bag count tolerance (keep exact if possible)
VAL_TOL  = 0
TEST_TOL = 0

# Optional: prior shift control (JS divergence of bag-level class priors)
USE_JS_CHECK = True
MAX_JS_TRAIN_TEST = 0.03   # very loose; seed4 had ~0.005

# Optional: enforce that val/test contain SOME multi-bag donors (avoid "train-only heavy-tail")
# If your data structurally cannot satisfy this, set to 0.
MIN_MULTI_BAG_DONORS_VAL  = 0
MIN_MULTI_BAG_DONORS_TEST = 0
MULTI_BAG_THRESHOLD = 2   # donor with >=2 bags is "multi-bag"

# Search
N_SEEDS_WANTED = 20
MAX_TRIES_PER_SEED = 20000
SEEDS = [1, 4, 7, 12, 14, 15, 21, 25, 29, 37, 44, 51, 63, 77, 88, 92, 101, 110, 123, 145]

# ------------------------
# 2) Helpers
# ------------------------
def quick_fingerprint_sha1(path, nbytes=8*1024*1024):
    h = hashlib.sha1()
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        h.update(f.read(nbytes))
        if size > nbytes:
            f.seek(max(0, size - nbytes))
            h.update(f.read(nbytes))
    return h.hexdigest(), size

def _write_list(path, items):
    with open(path, "w") as f:
        f.write("\n".join(map(str, items)) + "\n")

def js_divergence(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return 0.5 * (kl_pm + kl_qm)

def class_counts_from_donors(donors, donor2classcnt):
    out = {"control": 0, "mild/moderate": 0, "severe/critical": 0}
    for d in donors:
        c = donor2classcnt[d]
        out["control"] += c["control"]
        out["mild/moderate"] += c["mild/moderate"]
        out["severe/critical"] += c["severe/critical"]
    return out

def bags_from_donors(donors, donor2bags):
    return sum(donor2bags[d] for d in donors)

def donor_multi_count(donors, donor2bags, thr=2):
    return sum(1 for d in donors if donor2bags[d] >= thr)

def ok_minima(split_name, donors, donor2bags, donor2classcnt):
    cnt = class_counts_from_donors(donors, donor2classcnt)
    if split_name == "train":
        mins = MIN_PER_CLASS_TRAIN
        if any(cnt[k] < mins[k] for k in mins): return False
        return True
    if split_name == "val":
        mins = MIN_PER_CLASS_VAL
        if len(donors) < MIN_DONORS_VAL: return False
        if any(cnt[k] < mins[k] for k in mins): return False
        if MIN_MULTI_BAG_DONORS_VAL > 0:
            if donor_multi_count(donors, donor2bags, MULTI_BAG_THRESHOLD) < MIN_MULTI_BAG_DONORS_VAL:
                return False
        return True
    if split_name == "test":
        mins = MIN_PER_CLASS_TEST
        if len(donors) < MIN_DONORS_TEST: return False
        if any(cnt[k] < mins[k] for k in mins): return False
        if MIN_MULTI_BAG_DONORS_TEST > 0:
            if donor_multi_count(donors, donor2bags, MULTI_BAG_THRESHOLD) < MIN_MULTI_BAG_DONORS_TEST:
                return False
        return True
    raise ValueError(split_name)

def bag_prior(donors, donor2classcnt, order=("control","mild/moderate","severe/critical")):
    cnt = class_counts_from_donors(donors, donor2classcnt)
    v = np.array([cnt[k] for k in order], dtype=float)
    if v.sum() == 0:
        return v
    return v / v.sum()

# ------------------------
# 3) Load Data (safe copy)
# ------------------------
print("=== SECURE DATA LOAD ===")
if os.path.exists(LOCAL_PATH):
    os.remove(LOCAL_PATH)
print(f"Copying from Drive: {DRIVE_PATH} -> {LOCAL_PATH}")
shutil.copy(DRIVE_PATH, LOCAL_PATH)

fp_sha1, file_size = quick_fingerprint_sha1(LOCAL_PATH)
print(f"Fingerprint SHA1(head+tail): {fp_sha1}")
print(f"File size: {round(file_size/1024**3, 3)} GB")

adata = ad.read_h5ad(LOCAL_PATH, backed="r")
obs = adata.obs[[COL_SAMPLE, COL_DONOR, COL_Y]].copy()
for c in [COL_SAMPLE, COL_DONOR, COL_Y]:
    obs[c] = obs[c].astype(str)

# bag-level
g = obs.groupby(COL_SAMPLE, sort=False)
sample_tbl = pd.DataFrame({
    "donor_id": g[COL_DONOR].first(),
    "y":       g[COL_Y].first(),
    "n_cells": g.size()
})
# integrity
assert (g[COL_DONOR].nunique() == 1).all(), "CRITICAL: sampleID->donor_id not single-valued"
assert (g[COL_Y].nunique() == 1).all(), "CRITICAL: sampleID->severity not single-valued"

# donor stats
donor_grp = sample_tbl.groupby("donor_id", sort=False)
# donor2sids = donor_grp.apply(lambda df: df.index.tolist()).to_dict()
# ↓ 대체 (빠름, warning 없음)
donor2sids = (
    sample_tbl.reset_index()
    .groupby("donor_id", sort=False)["sampleID"]
    .agg(list)
    .to_dict()
)
donor2bags = {d: len(v) for d, v in donor2sids.items()}

donor2bags = {d: len(sids) for d, sids in donor2sids.items()}

# donor class bag counts
classes = ["control", "mild/moderate", "severe/critical"]
donor2classcnt = {}
for d, df in donor_grp:
    vc = df["y"].value_counts()
    donor2classcnt[d] = {k: int(vc.get(k, 0)) for k in classes}

all_donors = list(donor2sids.keys())
n_total_bags = sample_tbl.shape[0]

n_train_tgt = int(round(n_total_bags * R_TRAIN))
n_val_tgt   = int(round(n_total_bags * R_VAL))
n_test_tgt  = n_total_bags - n_train_tgt - n_val_tgt

print(f"Total bags={n_total_bags} donors={len(all_donors)} targets: train={n_train_tgt} val={n_val_tgt} test={n_test_tgt}")
global_cnt = sample_tbl["y"].value_counts().to_dict()
print("Global bag counts:", global_cnt)

# ------------------------
# 4) Core Splitter (Balanced Assignment + Repair)
# ------------------------
def balanced_initial_assignment(rng, donors, donor2bags):
    """
    Place large donors first, always into the split with the largest bag-deficit.
    Random tie-break for donors with same bagcount.
    """
    # donors sorted by bagcount desc, tie random
    donors = list(donors)
    rng.shuffle(donors)
    donors.sort(key=lambda d: donor2bags[d], reverse=True)

    splits = {"train": [], "val": [], "test": []}
    tgt = {"train": n_train_tgt, "val": n_val_tgt, "test": n_test_tgt}
    cur = {"train": 0, "val": 0, "test": 0}

    for d in donors:
        # choose split with max deficit (target-cur), but do not exceed by too much if avoidable
        deficits = {s: (tgt[s] - cur[s]) for s in tgt}
        # prefer splits still under target; if all over, choose smallest overflow
        under = [s for s in deficits if deficits[s] > 0]
        if under:
            # maximize deficit; tie random via rng
            best = sorted(under, key=lambda s: (deficits[s], rng.random()), reverse=True)[0]
        else:
            # all overflow: choose minimal overflow (closest to target)
            best = sorted(deficits.keys(), key=lambda s: (abs(deficits[s]), rng.random()))[0]

        splits[best].append(d)
        cur[best] += donor2bags[d]

    return splits

def repair_counts_by_moves(rng, splits, donor2bags, max_steps=5000):
    """
    Fix bagcount mismatch by moving donors from surplus split to deficit split.
    (donor atomic)
    """
    tgt = {"train": n_train_tgt, "val": n_val_tgt, "test": n_test_tgt}

    def cur_counts():
        return {s: bags_from_donors(splits[s], donor2bags) for s in splits}

    for _ in range(max_steps):
        cur = cur_counts()
        delta = {s: (cur[s] - tgt[s]) for s in splits}  # surplus positive
        if abs(delta["val"]) <= VAL_TOL and abs(delta["test"]) <= TEST_TOL and delta["train"] == 0:
            return True

        # pick a deficit split and surplus split
        deficit_splits = [s for s in splits if delta[s] < 0]
        surplus_splits = [s for s in splits if delta[s] > 0]
        if not deficit_splits or not surplus_splits:
            # cannot fix
            return False

        dst = sorted(deficit_splits, key=lambda s: (delta[s], rng.random()))[0]  # most negative
        src = sorted(surplus_splits, key=lambda s: (delta[s], rng.random()), reverse=True)[0]  # most positive

        need = -delta[dst]
        # choose donor in src to move that best matches need (bagcount close), tie random
        cand = splits[src]
        if not cand:
            return False

        # restrict: do not empty src completely (avoid pathological)
        if len(cand) <= 1:
            return False

        d_move = sorted(cand, key=lambda d: (abs(donor2bags[d] - need), rng.random()))[0]
        splits[src].remove(d_move)
        splits[dst].append(d_move)

    return False

def try_swap_fix(rng, splits, donor2bags, donor2classcnt, n_swaps=2000):
    """
    If minima fail, try donor swaps between splits to satisfy minima while keeping exact bag targets.
    """
    tgt = {"train": n_train_tgt, "val": n_val_tgt, "test": n_test_tgt}
    def cur_counts():
        return {s: bags_from_donors(splits[s], donor2bags) for s in splits}

    for _ in range(n_swaps):
        # pick two splits and donors
        a, b = rng.choice(["train","val","test"], size=2, replace=False)
        if not splits[a] or not splits[b]:
            continue
        da = rng.choice(splits[a])
        db = rng.choice(splits[b])

        # swap must preserve targets (since donor bagcounts differ)
        # so only swap donors with equal bagcount
        if donor2bags[da] != donor2bags[db]:
            continue

        # apply swap
        splits[a].remove(da); splits[b].remove(db)
        splits[a].append(db); splits[b].append(da)

        # keep exact bag targets (should hold)
        cur = cur_counts()
        if any(cur[s] != tgt[s] for s in cur):
            # revert
            splits[a].remove(db); splits[b].remove(da)
            splits[a].append(da); splits[b].append(db)
            continue

        # check minima
        if ok_minima("train", splits["train"], donor2bags, donor2classcnt) and \
           ok_minima("val",   splits["val"],   donor2bags, donor2classcnt) and \
           ok_minima("test",  splits["test"],  donor2bags, donor2classcnt):
            return True

        # if not good, maybe keep swap with small probability? (no, keep strict: revert)
        splits[a].remove(db); splits[b].remove(da)
        splits[a].append(da); splits[b].append(db)

    return False

def validate_split(splits, donor2bags, donor2classcnt):
    # exact counts
    c_train = bags_from_donors(splits["train"], donor2bags)
    c_val   = bags_from_donors(splits["val"], donor2bags)
    c_test  = bags_from_donors(splits["test"], donor2bags)
    if not (c_train == n_train_tgt and abs(c_val - n_val_tgt) <= VAL_TOL and abs(c_test - n_test_tgt) <= TEST_TOL):
        return False, "bagcount mismatch"

    # disjoint donors
    set_tr, set_va, set_te = set(splits["train"]), set(splits["val"]), set(splits["test"])
    if (set_tr & set_va) or (set_tr & set_te) or (set_va & set_te):
        return False, "donor leakage"

    # minima
    if not ok_minima("train", splits["train"], donor2bags, donor2classcnt):
        return False, "train minima fail"
    if not ok_minima("val", splits["val"], donor2bags, donor2classcnt):
        return False, "val minima fail"
    if not ok_minima("test", splits["test"], donor2bags, donor2classcnt):
        return False, "test minima fail"

    # optional JS check
    if USE_JS_CHECK:
        p_tr = bag_prior(splits["train"], donor2classcnt)
        p_te = bag_prior(splits["test"], donor2classcnt)
        js = js_divergence(p_tr, p_te)
        if js > MAX_JS_TRAIN_TEST:
            return False, f"JS(train,test) too high: {js:.4f}"

    return True, "ok"

# ------------------------
# 5) Search Loop
# ------------------------
os.makedirs(OUT_ROOT, exist_ok=True)
accepted = 0

print("\n=== STARTING SEARCH (BALANCED + REPAIR) ===")
print(f"Constraints: multi-bag donors val/test >= {MIN_MULTI_BAG_DONORS_VAL}/{MIN_MULTI_BAG_DONORS_TEST} (thr={MULTI_BAG_THRESHOLD})")
print(f"JS check: {USE_JS_CHECK} max_js_train_test={MAX_JS_TRAIN_TEST}")

for seed in SEEDS:
    if accepted >= N_SEEDS_WANTED:
        break

    rng = np.random.default_rng(seed)
    t0 = time.time()
    found = False

    for t in range(1, MAX_TRIES_PER_SEED + 1):
        # 1) balanced init
        splits = balanced_initial_assignment(rng, all_donors, donor2bags)

        # 2) repair bag counts (should already be close; force exact)
        ok_counts = repair_counts_by_moves(rng, splits, donor2bags, max_steps=3000)
        if not ok_counts:
            continue

        # 3) validate; if minima fail, try swaps (same bagcount swaps preserve targets)
        ok, why = validate_split(splits, donor2bags, donor2classcnt)
        if not ok:
            # attempt swap repair for minima/prior issues
            repaired = try_swap_fix(rng, splits, donor2bags, donor2classcnt, n_swaps=3000)
            if not repaired:
                continue
            ok, why = validate_split(splits, donor2bags, donor2classcnt)
            if not ok:
                continue

        # SUCCESS
        elapsed = round(time.time() - t0, 3)

        # expand to sampleIDs
        def expand(donors):
            sids = []
            for d in donors:
                sids.extend(donor2sids[d])
            return sorted(sids)

        d_train = sorted(splits["train"])
        d_val   = sorted(splits["val"])
        d_test  = sorted(splits["test"])
        s_train = expand(d_train)
        s_val   = expand(d_val)
        s_test  = expand(d_test)

        cnt_train = class_counts_from_donors(d_train, donor2classcnt)
        cnt_val   = class_counts_from_donors(d_val, donor2classcnt)
        cnt_test  = class_counts_from_donors(d_test, donor2classcnt)

        meta = {
            "seed": int(seed),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "source_file": {
                "drive_path": DRIVE_PATH,
                "local_copy": LOCAL_PATH,
                "fingerprint_sha1_headtail": fp_sha1,
                "size_bytes": file_size,
            },
            "targets": {"train": n_train_tgt, "val": n_val_tgt, "test": n_test_tgt},
            "tolerances": {"val_tol": VAL_TOL, "test_tol": TEST_TOL},
            "constraints": {
                "min_per_class_train": MIN_PER_CLASS_TRAIN,
                "min_per_class_val": MIN_PER_CLASS_VAL,
                "min_per_class_test": MIN_PER_CLASS_TEST,
                "min_donors_val": MIN_DONORS_VAL,
                "min_donors_test": MIN_DONORS_TEST,
                "min_multi_bag_donors_val": MIN_MULTI_BAG_DONORS_VAL,
                "min_multi_bag_donors_test": MIN_MULTI_BAG_DONORS_TEST,
                "multi_bag_threshold": MULTI_BAG_THRESHOLD,
                "use_js_check": USE_JS_CHECK,
                "max_js_train_test": MAX_JS_TRAIN_TEST,
            },
            "stats": {
                "bags": {"train": len(s_train), "val": len(s_val), "test": len(s_test)},
                "donors": {"train": len(d_train), "val": len(d_val), "test": len(d_test)},
                "class_counts_bags": {"train": cnt_train, "val": cnt_val, "test": cnt_test},
                "multi_bag_donors": {
                    "train": donor_multi_count(d_train, donor2bags, MULTI_BAG_THRESHOLD),
                    "val":   donor_multi_count(d_val,   donor2bags, MULTI_BAG_THRESHOLD),
                    "test":  donor_multi_count(d_test,  donor2bags, MULTI_BAG_THRESHOLD),
                },
            },
            "search": {"tries": t, "elapsed_sec": elapsed},
        }

        out_dir = os.path.join(OUT_ROOT, f"seed{seed}")
        os.makedirs(out_dir, exist_ok=True)

        _write_list(os.path.join(out_dir, "donors_train.txt"), d_train)
        _write_list(os.path.join(out_dir, "donors_val.txt"),   d_val)
        _write_list(os.path.join(out_dir, "donors_test.txt"),  d_test)

        _write_list(os.path.join(out_dir, "samples_train.txt"), s_train)
        _write_list(os.path.join(out_dir, "samples_val.txt"),   s_val)
        _write_list(os.path.join(out_dir, "samples_test.txt"),  s_test)

        with open(os.path.join(out_dir, "meta.json"), "w") as f:
            json.dump(meta, f, indent=2)

        accepted += 1
        print(
            f"[ACCEPT] seed={seed} tries={t} {elapsed}s | "
            f"bags(train/val/test)={len(s_train)}/{len(s_val)}/{len(s_test)} | "
            f"donors(train/val/test)={len(d_train)}/{len(d_val)}/{len(d_test)} | "
            f"ctrl(train/val/test)={cnt_train['control']}/{cnt_val['control']}/{cnt_test['control']} | "
            f"multi(val/test)={meta['stats']['multi_bag_donors']['val']}/{meta['stats']['multi_bag_donors']['test']}"
        )

        found = True
        break

    if not found:
        print(f"[FAIL] seed={seed}: no solution within {MAX_TRIES_PER_SEED} tries")

print(f"\n[DONE] accepted={accepted}/{N_SEEDS_WANTED} OUT_ROOT={OUT_ROOT}")


=== SECURE DATA LOAD ===
Copying from Drive: /content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad -> /content/temp_data_hvg4000.h5ad
Fingerprint SHA1(head+tail): 26e227677d263d81a8d991ac98192dddbd769f71
File size: 2.351 GB
Total bags=282 donors=196 targets: train=197 val=28 test=57
Global bag counts: {'severe/critical': 132, 'mild/moderate': 122, 'control': 28}

=== STARTING SEARCH (BALANCED + REPAIR) ===
Constraints: multi-bag donors val/test >= 0/0 (thr=2)
JS check: True max_js_train_test=0.03
[ACCEPT] seed=1 tries=5 1.727s | bags(train/val/test)=197/28/57 | donors(train/val/test)=111/28/57 | ctrl(train/val/test)=15/4/9 | multi(val/test)=0/0
[ACCEPT] seed=4 tries=2 0.436s | bags(train/val/test)=197/28/57 | donors(train/val/test)=111/28/57 | ctrl(train/val/test)=15/4/9 | multi(val/test)=0/0
[ACCEPT] seed=7 tries=2 0.479s | bags(train/val/test)=197/28/57 | donors(train/val/test)=111/28/57 | ctrl(train/val/test)=16/5/7 | multi(val

In [15]:
import os, json, glob, hashlib
import numpy as np
import pandas as pd
import anndata as ad

# ------------------------
# Config
# ------------------------
ADATA_PATH = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"
OUT_ROOT   = "/content/drive/MyDrive/scMILD_residual/splits_donor_holdout"

COL_SAMPLE = "sampleID"
COL_DONOR  = "donor_id"
COL_Y      = "CoVID-19 severity"

EXPECTED_Y = {"control", "mild/moderate", "severe/critical"}
TARGETS = {"train": 197, "val": 28, "test": 57}

# Hard minimums you enforced (edit if you changed)
MIN_CTRL_TRAIN = 15
MIN_CTRL_VAL   = 4
MIN_CTRL_TEST  = 6

# ------------------------
# Helpers
# ------------------------
def read_list(path):
    with open(path, "r") as f:
        xs = [line.strip() for line in f if line.strip()]
    return xs

def js_divergence(p, q, eps=1e-12):
    # p, q are dicts over same keys
    keys = sorted(set(p) | set(q))
    P = np.array([p.get(k, 0.0) for k in keys], dtype=float) + eps
    Q = np.array([q.get(k, 0.0) for k in keys], dtype=float) + eps
    P /= P.sum(); Q /= Q.sum()
    M = 0.5*(P+Q)
    kl_pm = np.sum(P*np.log(P/M))
    kl_qm = np.sum(Q*np.log(Q/M))
    return 0.5*(kl_pm + kl_qm)

def split_donor_shape(donor2bags, split_donors):
    # returns summary stats of donor bagcount distribution in split
    arr = np.array([donor2bags[d] for d in split_donors], dtype=int) if split_donors else np.array([], dtype=int)
    if len(arr) == 0:
        return dict(n_donors=0, mean=np.nan, median=np.nan, max=np.nan, p95=np.nan,
                    frac_1bag=np.nan, frac_2plus=np.nan, top10pct_share=np.nan, top5_share=np.nan)
    arr_sorted = np.sort(arr)[::-1]
    n = len(arr_sorted)
    top10 = max(1, int(np.ceil(0.10*n)))
    top5  = min(5, n)
    return dict(
        n_donors=n,
        mean=float(arr.mean()),
        median=float(np.median(arr)),
        max=int(arr.max()),
        p95=float(np.percentile(arr, 95)),
        frac_1bag=float((arr == 1).mean()),
        frac_2plus=float((arr >= 2).mean()),
        top10pct_share=float(arr_sorted[:top10].sum() / arr_sorted.sum()),
        top5_share=float(arr_sorted[:top5].sum() / arr_sorted.sum()),
    )

# ------------------------
# Build ground-truth sample table from adata
# ------------------------
print("Loading obs (backed) ...")
adata = ad.read_h5ad(ADATA_PATH, backed="r")
obs = adata.obs[[COL_SAMPLE, COL_DONOR, COL_Y]].copy()
for c in [COL_SAMPLE, COL_DONOR, COL_Y]:
    obs[c] = obs[c].astype(str)

# Per-sample truth
g = obs.groupby(COL_SAMPLE, sort=False)
sample_tbl = pd.DataFrame({
    "donor_id": g[COL_DONOR].first(),
    "y": g[COL_Y].first(),
    "n_cells": g.size(),
})

# Integrity (should already be true, but audit anyway)
assert (g[COL_DONOR].nunique() == 1).all(), "FAIL: sampleID->donor_id not single-valued in data"
assert (g[COL_Y].nunique() == 1).all(), "FAIL: sampleID->y not single-valued in data"
assert set(sample_tbl["y"].unique()).issubset(EXPECTED_Y), f"FAIL: unexpected y values: {sample_tbl['y'].unique()}"

all_sids = set(sample_tbl.index.astype(str))
sid2donor = sample_tbl["donor_id"].to_dict()
sid2y     = sample_tbl["y"].to_dict()

donor2sids = sample_tbl.reset_index().groupby("donor_id", sort=False)["sampleID"].agg(list).to_dict()
donor2bags = {d: len(v) for d, v in donor2sids.items()}

global_counts = sample_tbl["y"].value_counts().to_dict()
global_props  = {k: v/sum(global_counts.values()) for k,v in global_counts.items()}

# ------------------------
# Audit all seeds
# ------------------------
rows = []
seed_dirs = sorted([p for p in glob.glob(os.path.join(OUT_ROOT, "seed*")) if os.path.isdir(p)])

assert seed_dirs, f"No seed dirs found under {OUT_ROOT}"

for sd in seed_dirs:
    seed = int(os.path.basename(sd).replace("seed",""))

    # Load lists
    s_tr = read_list(os.path.join(sd, "samples_train.txt"))
    s_va = read_list(os.path.join(sd, "samples_val.txt"))
    s_te = read_list(os.path.join(sd, "samples_test.txt"))
    d_tr = read_list(os.path.join(sd, "donors_train.txt"))
    d_va = read_list(os.path.join(sd, "donors_val.txt"))
    d_te = read_list(os.path.join(sd, "donors_test.txt"))

    set_tr, set_va, set_te = set(s_tr), set(s_va), set(s_te)
    set_dtr, set_dva, set_dte = set(d_tr), set(d_va), set(d_te)

    # --- Core checks ---
    leak_samples = int(bool((set_tr & set_va) or (set_tr & set_te) or (set_va & set_te)))
    leak_donors  = int(bool((set_dtr & set_dva) or (set_dtr & set_dte) or (set_dva & set_dte)))

    union_ok = int(len(set_tr | set_va | set_te) == len(all_sids))
    missing  = len(all_sids - (set_tr | set_va | set_te))
    extra    = len((set_tr | set_va | set_te) - all_sids)

    # Targets
    tgt_ok = int((len(s_tr)==TARGETS["train"]) and (len(s_va)==TARGETS["val"]) and (len(s_te)==TARGETS["test"]))

    # Donor->sample consistency (every sample's donor must be in that split donor list)
    def donor_consistency(samples, donors):
        donors = set(donors)
        bad = [s for s in samples if sid2donor.get(s, None) not in donors]
        return len(bad)

    bad_tr = donor_consistency(s_tr, d_tr)
    bad_va = donor_consistency(s_va, d_va)
    bad_te = donor_consistency(s_te, d_te)

    # Class counts / mins
    def class_counts(sids):
        ys = [sid2y[s] for s in sids]
        vc = pd.Series(ys).value_counts()
        return {k:int(vc.get(k,0)) for k in EXPECTED_Y}

    cnt_tr = class_counts(s_tr)
    cnt_va = class_counts(s_va)
    cnt_te = class_counts(s_te)

    min_ok = int(
        (cnt_tr["control"] >= MIN_CTRL_TRAIN) and
        (cnt_va["control"] >= MIN_CTRL_VAL) and
        (cnt_te["control"] >= MIN_CTRL_TEST)
    )

    # Priors + JS
    def props(cnt):
        tot = sum(cnt.values())
        return {k: (cnt[k]/tot if tot else 0.0) for k in EXPECTED_Y}

    p_tr, p_va, p_te = props(cnt_tr), props(cnt_va), props(cnt_te)
    js_tr_va = js_divergence(p_tr, p_va)
    js_tr_te = js_divergence(p_tr, p_te)

    # Donor bagcount distribution shape
    sh_tr = split_donor_shape(donor2bags, d_tr)
    sh_va = split_donor_shape(donor2bags, d_va)
    sh_te = split_donor_shape(donor2bags, d_te)

    rows.append({
        "seed": seed,
        "bags_train": len(s_tr), "bags_val": len(s_va), "bags_test": len(s_te),
        "donors_train": len(d_tr), "donors_val": len(d_va), "donors_test": len(d_te),
        "leak_samples": leak_samples, "leak_donors": leak_donors,
        "union_ok": union_ok, "missing": missing, "extra": extra,
        "tgt_ok": tgt_ok,
        "bad_donor_map_train": bad_tr, "bad_donor_map_val": bad_va, "bad_donor_map_test": bad_te,
        "ctrl_train": cnt_tr["control"], "ctrl_val": cnt_va["control"], "ctrl_test": cnt_te["control"],
        "min_ok": min_ok,
        "js_train_val": js_tr_va, "js_train_test": js_tr_te,
        "train_max_bags_per_donor": sh_tr["max"],
        "train_top10pct_share": sh_tr["top10pct_share"],
        "val_max_bags_per_donor": sh_va["max"],
        "test_max_bags_per_donor": sh_te["max"],
    })

df = pd.DataFrame(rows).sort_values("seed").reset_index(drop=True)

# Summary flags
df["PASS_CORE"] = (
    (df["leak_samples"]==0) & (df["leak_donors"]==0) &
    (df["union_ok"]==1) & (df["missing"]==0) & (df["extra"]==0) &
    (df["tgt_ok"]==1) &
    (df["bad_donor_map_train"]==0) & (df["bad_donor_map_val"]==0) & (df["bad_donor_map_test"]==0) &
    (df["min_ok"]==1)
).astype(int)

print("\n=== AUDIT TABLE (key columns) ===")
print(df[[
    "seed","PASS_CORE",
    "bags_train","bags_val","bags_test",
    "donors_train","donors_val","donors_test",
    "ctrl_train","ctrl_val","ctrl_test",
    "leak_samples","leak_donors","missing","extra",
    "js_train_val","js_train_test",
    "train_max_bags_per_donor","train_top10pct_share"
]].to_string(index=False))

print("\n=== FAILURES (if any) ===")
bad = df[df["PASS_CORE"]==0]
if len(bad)==0:
    print("ALL SEEDS PASS core invariants ✅")
else:
    print(bad.to_string(index=False))

print("\n=== DISTRIBUTION SUMMARY ===")
print("train ctrl min/median/max:", df["ctrl_train"].min(), df["ctrl_train"].median(), df["ctrl_train"].max())
print("JS(train,test) min/median/max:", df["js_train_test"].min(), df["js_train_test"].median(), df["js_train_test"].max())
print("train max bags/donor min/median/max:", df["train_max_bags_per_donor"].min(), df["train_max_bags_per_donor"].median(), df["train_max_bags_per_donor"].max())
print("train top10% share min/median/max:", df["train_top10pct_share"].min(), df["train_top10pct_share"].median(), df["train_top10pct_share"].max())


Loading obs (backed) ...

=== AUDIT TABLE (key columns) ===
 seed  PASS_CORE  bags_train  bags_val  bags_test  donors_train  donors_val  donors_test  ctrl_train  ctrl_val  ctrl_test  leak_samples  leak_donors  missing  extra  js_train_val  js_train_test  train_max_bags_per_donor  train_top10pct_share
    1          1         197        28         57           111          28           57          15         4          9             0            0        0      0      0.007193       0.010347                         6              0.269036
    4          1         197        28         57           111          28           57          15         4          9             0            0        0      0      0.005801       0.010931                         6              0.269036
    7          1         197        28         57           111          28           57          16         5          7             0            0        0      0      0.012723       0.005471                     

In [5]:
# ============================================================
# Donor-holdout splits (70/15/15), control-balanced, feasible
# - Bags = sample_id
# - Donor-holdout strictly enforced
# - Targets computed from ratio with remainder assigned to VAL/TEST first
# - Control minima symmetric for 70/15/15 (default: val=5, test=5)
# - Robust builder:
#     (1) Start with random donor partition (val/test sized by bags)
#     (2) Local-search k-swap to hit bag targets within tolerance
#     (3) Repair to satisfy control minima (val/test first, then train)
#     (4) Validate donor minima and counts
# - Auto-tolerance schedule: try TOL=2 then 3 then 4 then 5 (val/test only)
# ============================================================

import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc

# -------------------------
# Config
# -------------------------
ADATA_PATH = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"
OUT_ROOT   = "/content/drive/MyDrive/scMILD_residual/splits_donor_holdout"

SEEDS = list(range(20))
RATIO = {"train": 0.70, "val": 0.15, "test": 0.15}

# control minima (RECOMMENDED for 70/15/15)
MIN_CTRL = {"train": 15, "val": 5, "test": 5}  # if impossible, set val/test to 4

# donor minima (avoid over-constraining train)
MIN_DONORS = {"train": 90, "val": 25, "test": 25}

# tolerance schedule to try (val/test); train kept exact via repairs
TOL_SCHEDULE = [2, 3, 4, 5]

# search budget
MAX_ATTEMPTS_PER_SEED = 4000
TIME_LIMIT_SEC = 180

# local search parameters
LS_ITERS = 12000          # iterations per attempt
LS_STALL = 2000           # stop if no improvement for this many iters
KSWAP_PROB = 0.35         # probability of trying 2-for-1 / 1-for-2 swaps
RNG_JITTER = 1e-3


# -------------------------
# Build bag table
# -------------------------
def autodetect_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def build_bag_table(adata_path):
    ad = sc.read_h5ad(adata_path, backed="r")
    obs = ad.obs

    sample_col = autodetect_col(obs.columns, ["sample_id","sampleID","SampleID","sample"])
    donor_col  = autodetect_col(obs.columns, ["donor_id","donorID","DonorID","donor"])
    y_col      = autodetect_col(obs.columns, ["y_norm","y","severity","Severity","COVID_severity","CoVID-19 severity","clinical_status"])
    if sample_col is None or donor_col is None or y_col is None:
        raise KeyError(f"Missing cols: sample={sample_col}, donor={donor_col}, y={y_col}")

    df = obs[[sample_col, donor_col, y_col]].copy()
    df.columns = ["sample_id","donor_id","y_raw"]
    df["sample_id"] = df["sample_id"].astype(str)
    df["donor_id"]  = df["donor_id"].astype(str)
    df["y_raw"]     = df["y_raw"].astype(str)

    g = df.groupby("sample_id", sort=False)
    bag = g.agg(donor_id=("donor_id","first"),
                y_raw=("y_raw","first"),
                n_cells=("y_raw","size")).reset_index()

    # single-valued checks
    if (g["donor_id"].nunique() != 1).any():
        raise ValueError("sample_id -> donor_id not single-valued")
    if (g["y_raw"].nunique() != 1).any():
        raise ValueError("sample_id -> y not single-valued")

    y = bag["y_raw"].str.strip().str.lower().replace({"healthy":"control","ctrl":"control"})
    bag["y_norm"] = y
    bag["is_control"] = (bag["y_norm"] == "control").astype(int)
    return bag

bag_tbl = build_bag_table(ADATA_PATH)
N_BAGS = len(bag_tbl)
N_DONORS = bag_tbl["donor_id"].nunique()
print("[bags]", N_BAGS, "donors:", N_DONORS, "labels:", sorted(bag_tbl["y_norm"].unique().tolist()))

donor_tbl = bag_tbl.groupby("donor_id").agg(
    n_bags=("sample_id","nunique"),
    n_ctrl=("is_control","sum"),
).reset_index()

donors_all = donor_tbl["donor_id"].astype(str).values
donor2bags = dict(zip(donor_tbl["donor_id"].astype(str), donor_tbl["n_bags"].astype(int)))
donor2ctrl = dict(zip(donor_tbl["donor_id"].astype(str), donor_tbl["n_ctrl"].astype(int)))
donor2samples = bag_tbl.groupby("donor_id")["sample_id"].apply(list).to_dict()

control_donors = donor_tbl.loc[donor_tbl["n_ctrl"] > 0, "donor_id"].astype(str).values

# feasibility quick check on control bags
total_ctrl_bags = int(bag_tbl["is_control"].sum())
req_ctrl = MIN_CTRL["train"] + MIN_CTRL["val"] + MIN_CTRL["test"]
if total_ctrl_bags < req_ctrl:
    raise ValueError(f"Impossible control minima: total control bags={total_ctrl_bags} < required={req_ctrl}")


# -------------------------
# Targets from ratio, remainder -> VAL/TEST first
# -------------------------
def compute_targets_valtest_first(n_total, ratio):
    keys = ["train","val","test"]
    raw = {k: ratio[k] * n_total for k in keys}
    base = {k: int(np.floor(raw[k])) for k in keys}
    rem = n_total - sum(base.values())
    # distribute remainder to VAL/TEST first, then TRAIN, by fractional parts within that priority
    priority = ["val","test","train"]
    fracs = sorted(priority, key=lambda k: (raw[k]-base[k]), reverse=True)
    for i in range(rem):
        base[fracs[i % len(fracs)]] += 1
    assert sum(base.values()) == n_total
    return base

TARGET = compute_targets_valtest_first(N_BAGS, RATIO)
print("[TARGET]", TARGET, "sum:", sum(TARGET.values()))


# -------------------------
# Basic utilities
# -------------------------
def donors_to_samples(donor_list):
    s = []
    for d in donor_list:
        s.extend(donor2samples[d])
    return sorted(set(map(str, s)))

def count_bags_ctrl(donor_list):
    bags = sum(donor2bags[d] for d in donor_list)
    ctrl = sum(donor2ctrl[d] for d in donor_list)
    return bags, ctrl

def split_stats(split):
    return {k: count_bags_ctrl(v) for k,v in split.items()}

def accept(split, tol_val, tol_test):
    train, val, test = split["train"], split["val"], split["test"]
    bt, ct = count_bags_ctrl(train)
    bv, cv = count_bags_ctrl(val)
    bq, cq = count_bags_ctrl(test)

    # bag targets with tol
    if bt != TARGET["train"]:
        return False
    if not (TARGET["val"] - tol_val <= bv <= TARGET["val"] + tol_val):
        return False
    if not (TARGET["test"] - tol_test <= bq <= TARGET["test"] + tol_test):
        return False

    # donor minima
    if len(train) < MIN_DONORS["train"]: return False
    if len(val)   < MIN_DONORS["val"]:   return False
    if len(test)  < MIN_DONORS["test"]:  return False

    # control minima
    if ct < MIN_CTRL["train"]: return False
    if cv < MIN_CTRL["val"]:   return False
    if cq < MIN_CTRL["test"]:  return False

    # disjointness
    if set(train) & set(val):  return False
    if set(train) & set(test): return False
    if set(val) & set(test):   return False
    return True

def bag_deviation(split, tol_val, tol_test):
    bt,_ = count_bags_ctrl(split["train"])
    bv,_ = count_bags_ctrl(split["val"])
    bq,_ = count_bags_ctrl(split["test"])
    # train exact target, so deviation is absolute
    return abs(bt - TARGET["train"]) + abs(bv - TARGET["val"]) + abs(bq - TARGET["test"])

def control_deficit(split):
    _, ct = count_bags_ctrl(split["train"])
    _, cv = count_bags_ctrl(split["val"])
    _, cq = count_bags_ctrl(split["test"])
    d = 0
    d += max(0, MIN_CTRL["train"] - ct)
    d += max(0, MIN_CTRL["val"]   - cv)
    d += max(0, MIN_CTRL["test"]  - cq)
    return d


# -------------------------
# Initial random split (by donors, aiming for bag targets)
# -------------------------
def initial_partition(rng, tol_val, tol_test):
    # Start by constructing val and test by accumulating donors until near target
    remaining = list(donors_all)
    rng.shuffle(remaining)

    val = []
    test = []
    train = []

    # helper to fill a split to roughly target bags
    def fill(target, tol, acc):
        bags = 0
        i = 0
        while i < len(remaining) and bags < (target - tol):
            d = remaining[i]
            # take it
            acc.append(d)
            bags += donor2bags[d]
            remaining.pop(i)
            # don't increment i (list shrank)
        return bags

    # fill val/test first; train is remainder
    fill(TARGET["val"], tol_val, val)
    fill(TARGET["test"], tol_test, test)
    train = remaining[:]  # remainder

    # Now we need train exact TARGET["train"]; fix by moving donors between splits using local search.
    return {"train": train, "val": val, "test": test}


# -------------------------
# Local search moves (k-swaps)
# -------------------------
def move_one(split, src, dst, d):
    split[src].remove(d)
    split[dst].append(d)

def try_improve_bags(rng, split, tol_val, tol_test):
    """
    Improve bag deviation primarily (train exactness matters).
    Use 1-1 swaps and occasionally 2-for-1 / 1-for-2 swaps between train and val/test.
    """
    best_dev = bag_deviation(split, tol_val, tol_test)
    stall = 0

    for it in range(LS_ITERS):
        if stall > LS_STALL:
            break

        # if train already exact and val/test within tol, stop early
        bt,_ = count_bags_ctrl(split["train"])
        bv,_ = count_bags_ctrl(split["val"])
        bq,_ = count_bags_ctrl(split["test"])
        if bt == TARGET["train"] and (TARGET["val"]-tol_val <= bv <= TARGET["val"]+tol_val) and (TARGET["test"]-tol_test <= bq <= TARGET["test"]+tol_test):
            return True

        # choose which pair to operate on
        pair = rng.choice(["train-val","train-test","val-test"])
        a, b = pair.split("-")

        # propose swap types
        use_kswap = (rng.random() < KSWAP_PROB)

        def bags_of(lst): return sum(donor2bags[d] for d in lst)

        if not split[a] or not split[b]:
            stall += 1
            continue

        # Current deviation
        dev0 = best_dev

        # 1-1 swap
        if not use_kswap:
            da = rng.choice(split[a])
            db = rng.choice(split[b])

            # simulate swap
            # compute new bag totals quickly
            bt,_ = count_bags_ctrl(split["train"])
            bv,_ = count_bags_ctrl(split["val"])
            bq,_ = count_bags_ctrl(split["test"])

            new = {"train": bt, "val": bv, "test": bq}
            new[a] = new[a] - donor2bags[da] + donor2bags[db]
            new[b] = new[b] - donor2bags[db] + donor2bags[da]

            # enforce caps for val/test (train must end exact later; allow intermediate)
            if new["val"] > TARGET["val"] + tol_val or new["val"] < TARGET["val"] - tol_val - 10:
                stall += 1
                continue
            if new["test"] > TARGET["test"] + tol_test or new["test"] < TARGET["test"] - tol_test - 10:
                stall += 1
                continue

            dev1 = abs(new["train"]-TARGET["train"]) + abs(new["val"]-TARGET["val"]) + abs(new["test"]-TARGET["test"])
            if dev1 + rng.random()*RNG_JITTER < dev0:
                # commit
                split[a].remove(da); split[b].remove(db)
                split[a].append(db); split[b].append(da)
                best_dev = dev1
                stall = 0
            else:
                stall += 1

        # 2-for-1 or 1-for-2 swap
        else:
            # decide direction
            if len(split[a]) < 2:
                stall += 1
                continue

            # choose 2 donors from a, 1 from b
            da1, da2 = rng.choice(split[a], size=2, replace=False)
            db = rng.choice(split[b])

            bt,_ = count_bags_ctrl(split["train"])
            bv,_ = count_bags_ctrl(split["val"])
            bq,_ = count_bags_ctrl(split["test"])
            new = {"train": bt, "val": bv, "test": bq}

            new[a] = new[a] - donor2bags[da1] - donor2bags[da2] + donor2bags[db]
            new[b] = new[b] - donor2bags[db] + donor2bags[da1] + donor2bags[da2]

            if new["val"] > TARGET["val"] + tol_val or new["val"] < TARGET["val"] - tol_val - 10:
                stall += 1
                continue
            if new["test"] > TARGET["test"] + tol_test or new["test"] < TARGET["test"] - tol_test - 10:
                stall += 1
                continue

            dev1 = abs(new["train"]-TARGET["train"]) + abs(new["val"]-TARGET["val"]) + abs(new["test"]-TARGET["test"])
            if dev1 + rng.random()*RNG_JITTER < dev0:
                # commit
                split[a].remove(da1); split[a].remove(da2)
                split[b].remove(db)
                split[a].append(db)
                split[b].append(da1); split[b].append(da2)
                best_dev = dev1
                stall = 0
            else:
                stall += 1

    return False


def repair_controls(rng, split, tol_val, tol_test):
    """
    Ensure control minima by moving control donors into deficit splits.
    Priority: test, val, then train.
    Keep bag targets within tolerance by swapping with similarly-sized donors.
    """
    def bags(splitname):
        return count_bags_ctrl(split[splitname])[0]

    def ctrl(splitname):
        return count_bags_ctrl(split[splitname])[1]

    # helper: find donor in src with ctrl>0
    def pick_ctrl_donor(src):
        cands = [d for d in split[src] if donor2ctrl[d] > 0]
        if not cands:
            return None
        return rng.choice(cands)

    # helper: find donor in dst to swap out (prefer same bag size, preferably non-control)
    def pick_swap_out(dst, want_bagsize):
        cands = split[dst]
        # prioritize non-control with same bag size, then any with same size, then any
        same = [d for d in cands if donor2bags[d] == want_bagsize and donor2ctrl[d] == 0]
        if same:
            return rng.choice(same)
        same2 = [d for d in cands if donor2bags[d] == want_bagsize]
        if same2:
            return rng.choice(same2)
        return rng.choice(cands)

    # attempt to fix deficits iteratively
    for _ in range(4000):
        deficits = {
            "test": max(0, MIN_CTRL["test"] - ctrl("test")),
            "val":  max(0, MIN_CTRL["val"]  - ctrl("val")),
            "train":max(0, MIN_CTRL["train"]- ctrl("train")),
        }
        if deficits["test"] == 0 and deficits["val"] == 0 and deficits["train"] == 0:
            return True

        # fix test first, then val, then train
        if deficits["test"] > 0:
            need = "test"
        elif deficits["val"] > 0:
            need = "val"
        else:
            need = "train"

        # source: choose from splits with extra controls
        src_candidates = ["train","val","test"]
        src_candidates.remove(need)
        rng.shuffle(src_candidates)

        moved = False
        for src in src_candidates:
            d_in = pick_ctrl_donor(src)
            if d_in is None:
                continue

            # swap d_in into need, swap someone out of need to keep bags stable
            d_out = pick_swap_out(need, donor2bags[d_in])

            # simulate swap bag totals
            bt,_ = count_bags_ctrl(split["train"])
            bv,_ = count_bags_ctrl(split["val"])
            bq,_ = count_bags_ctrl(split["test"])
            cur = {"train": bt, "val": bv, "test": bq}

            new = cur.copy()
            new[src]  = new[src]  - donor2bags[d_in] + donor2bags[d_out]
            new[need] = new[need] - donor2bags[d_out] + donor2bags[d_in]

            # check tolerances (train exactness can be repaired later, but don't let it explode)
            if abs(new["train"] - TARGET["train"]) > 10:
                continue
            if not (TARGET["val"]-tol_val <= new["val"] <= TARGET["val"]+tol_val):
                continue
            if not (TARGET["test"]-tol_test <= new["test"] <= TARGET["test"]+tol_test):
                continue

            # commit swap
            split[src].remove(d_in); split[need].remove(d_out)
            split[src].append(d_out); split[need].append(d_in)
            moved = True
            break

        if not moved:
            return False

    return False


# -------------------------
# Build one split attempt
# -------------------------
def build_one(rng, tol_val, tol_test):
    split = initial_partition(rng, tol_val, tol_test)

    # local search: fix bag targets (esp. train exact)
    ok_bags = try_improve_bags(rng, split, tol_val, tol_test)
    if not ok_bags:
        return None

    # repair controls under bag tolerances
    ok_ctrl = repair_controls(rng, split, tol_val, tol_test)
    if not ok_ctrl:
        return None

    # final tighten bag targets again
    ok_bags2 = try_improve_bags(rng, split, tol_val, tol_test)
    if not ok_bags2:
        return None

    # Final accept
    if not accept(split, tol_val, tol_test):
        return None

    return split


# -------------------------
# Save outputs
# -------------------------
def save_seed(out_root: Path, seed: int, attempt: int, dt: float, tol_val: int, tol_test: int, split: dict):
    out_root.mkdir(parents=True, exist_ok=True)
    seed_dir = out_root / f"seed{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    split_samples = {s: donors_to_samples(split[s]) for s in ["train","val","test"]}

    for s in ["train","val","test"]:
        (seed_dir / f"donors_{s}.txt").write_text("\n".join(split[s]) + "\n")
        (seed_dir / f"samples_{s}.txt").write_text("\n".join(split_samples[s]) + "\n")

    bt, ct = count_bags_ctrl(split["train"])
    bv, cv = count_bags_ctrl(split["val"])
    bq, cq = count_bags_ctrl(split["test"])

    meta = {
        "seed": seed,
        "attempt": attempt,
        "time_sec": round(dt, 3),
        "method": "OptionA_localSearch_kSwap_controlRepair",
        "ratio": RATIO,
        "targets": TARGET,
        "tolerance": {"train": 0, "val": tol_val, "test": tol_test},
        "min_controls": MIN_CTRL,
        "min_donors": MIN_DONORS,
        "bag_counts": {"train": bt, "val": bv, "test": bq},
        "control_counts": {"train": ct, "val": cv, "test": cq},
        "n_donors": {s: len(split[s]) for s in ["train","val","test"]},
        "n_samples": {s: len(split_samples[s]) for s in ["train","val","test"]},
    }
    (seed_dir / "meta.json").write_text(json.dumps(meta, indent=2) + "\n")

    print(f"[OK] seed{seed} attempt={attempt} tolV={tol_val} tolT={tol_test} "
          f"bags={{train:{bt}, val:{bv}, test:{bq}}} "
          f"ctrl={{train:{ct}, val:{cv}, test:{cq}}} "
          f"donors={{train:{len(split['train'])}, val:{len(split['val'])}, test:{len(split['test'])}}} "
          f"time={dt:.2f}s")


# -------------------------
# Main loop: per seed, try increasing tolerances automatically
# -------------------------
def generate_all():
    out_root = Path(OUT_ROOT)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        t0 = time.time()
        ok = False
        attempt = 0

        for tol in TOL_SCHEDULE:
            tol_val = tol
            tol_test = tol
            # Try multiple attempts per tolerance
            for _ in range(MAX_ATTEMPTS_PER_SEED):
                attempt += 1
                if (time.time() - t0) > TIME_LIMIT_SEC:
                    break
                split = build_one(rng, tol_val, tol_test)
                if split is not None:
                    save_seed(out_root, seed, attempt, time.time()-t0, tol_val, tol_test, split)
                    ok = True
                    break
            if ok or (time.time() - t0) > TIME_LIMIT_SEC:
                break

        if not ok:
            print(f"[FAIL] seed{seed} no valid split within budget (time={time.time()-t0:.1f}s). "
                  f"Try lowering MIN_DONORS(train) or MIN_CTRL(val/test) to 4/4.")

generate_all()
print("DONE:", OUT_ROOT)


[bags] 282 donors: 196 labels: ['control', 'mild/moderate', 'severe/critical']
[TARGET] {'train': 198, 'val': 42, 'test': 42} sum: 282
[OK] seed0 attempt=1 tolV=2 tolT=2 bags={train:198, val:42, test:42} ctrl={train:18, val:5, test:5} donors={train:129, val:32, test:35} time=0.01s
[OK] seed1 attempt=1 tolV=2 tolT=2 bags={train:198, val:40, test:44} ctrl={train:15, val:5, test:8} donors={train:138, val:26, test:32} time=0.01s
[OK] seed2 attempt=1 tolV=2 tolT=2 bags={train:198, val:43, test:41} ctrl={train:18, val:5, test:5} donors={train:134, val:33, test:29} time=0.01s
[OK] seed3 attempt=1 tolV=2 tolT=2 bags={train:198, val:42, test:42} ctrl={train:18, val:5, test:5} donors={train:140, val:29, test:27} time=0.02s
[OK] seed4 attempt=1 tolV=2 tolT=2 bags={train:198, val:43, test:41} ctrl={train:17, val:6, test:5} donors={train:140, val:27, test:29} time=0.02s
[OK] seed5 attempt=1 tolV=2 tolT=2 bags={train:198, val:42, test:42} ctrl={train:18, val:5, test:5} donors={train:133, val:31, tes

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import scanpy as sc

ADATA_PATH = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"
SPLIT_ROOT = "/content/drive/MyDrive/scMILD_residual/splits_donor_holdout"

TARGET = {"train": 198, "val": 42, "test": 42}
TOL    = {"train": 0,   "val": 2,  "test": 2}
MIN_CTRL = {"train": 15, "val": 5, "test": 5}

def autodetect_col(cols, candidates):
    for c in candidates:
        if c in cols: return c
    return None

def jsd(p, q, eps=1e-12):
    p = np.asarray(p, float); q = np.asarray(q, float)
    p = p / (p.sum() + eps); q = q / (q.sum() + eps)
    m = 0.5*(p+q)
    def kl(a,b):
        a = np.clip(a, eps, 1.0); b = np.clip(b, eps, 1.0)
        return float(np.sum(a*np.log(a/b)))
    return 0.5*kl(p,m) + 0.5*kl(q,m)

def ks_distance(a, b):
    a = np.sort(np.asarray(a)); b = np.sort(np.asarray(b))
    if len(a)==0 or len(b)==0: return np.nan
    vals = np.unique(np.concatenate([a,b]))
    fa = np.searchsorted(a, vals, side="right")/len(a)
    fb = np.searchsorted(b, vals, side="right")/len(b)
    return float(np.max(np.abs(fa-fb)))

# ---- bag table from obs ----
ad = sc.read_h5ad(ADATA_PATH, backed="r")
obs = ad.obs

sample_col = autodetect_col(obs.columns, ["sample_id","sampleID","SampleID","sample"])
donor_col  = autodetect_col(obs.columns, ["donor_id","donorID","DonorID","donor"])
y_col      = autodetect_col(obs.columns, ["y_norm","y","severity","Severity","COVID_severity","CoVID-19 severity","clinical_status"])
if sample_col is None or donor_col is None or y_col is None:
    raise KeyError("missing cols")

df = obs[[sample_col, donor_col, y_col]].copy()
df.columns = ["sample_id","donor_id","y"]
df["sample_id"] = df["sample_id"].astype(str)
df["donor_id"]  = df["donor_id"].astype(str)
df["y"]         = df["y"].astype(str).str.strip().str.lower().replace({"healthy":"control","ctrl":"control"})

bag_tbl = df.groupby("sample_id", sort=False).agg(
    donor_id=("donor_id","first"),
    y=("y","first"),
    n_cells=("y","size"),
).reset_index()
bag_tbl["is_control"] = (bag_tbl["y"]=="control").astype(int)

labels = sorted(bag_tbl["y"].unique().tolist())
global_counts = bag_tbl["y"].value_counts().reindex(labels, fill_value=0).values
global_p = global_counts / global_counts.sum()
donor_bags_global = bag_tbl.groupby("donor_id")["sample_id"].nunique().values

# ---- iterate seeds ----
seed_dirs = sorted([p for p in Path(SPLIT_ROOT).glob("seed*") if p.is_dir()],
                   key=lambda p: int(p.name.replace("seed","")))

rows = []
for sd in seed_dirs:
    seed = int(sd.name.replace("seed",""))
    split_samples = {}
    for sp in ["train","val","test"]:
        split_samples[sp] = [x.strip() for x in (sd/f"samples_{sp}.txt").read_text().splitlines() if x.strip()]

    m = {"seed": seed, "violations": 0}

    # compute per split metrics
    for sp in ["train","val","test"]:
        sids = set(split_samples[sp])
        sub = bag_tbl[bag_tbl["sample_id"].isin(sids)]

        bags = int(sub.shape[0])
        donors = int(sub["donor_id"].nunique())
        ctrls = int(sub["is_control"].sum())

        m[f"bags_{sp}"] = bags
        m[f"donors_{sp}"] = donors
        m[f"controls_{sp}"] = ctrls

        # hard constraint check
        if sp=="train":
            if bags != TARGET[sp]: m["violations"] += 1
        else:
            if not (TARGET[sp]-TOL[sp] <= bags <= TARGET[sp]+TOL[sp]): m["violations"] += 1
        if ctrls < MIN_CTRL[sp]: m["violations"] += 1

        # label JSD
        c = sub["y"].value_counts().reindex(labels, fill_value=0).values
        p = c / max(c.sum(), 1)
        m[f"jsd_label_{sp}"] = jsd(p, global_p)

        # donor bag KS vs global
        donor_bags_split = sub.groupby("donor_id")["sample_id"].nunique().values
        m[f"ks_donorbag_{sp}"] = ks_distance(donor_bags_split, donor_bags_global)

        # donor domination
        max_b = int(donor_bags_split.max()) if len(donor_bags_split) else 0
        m[f"max_donor_frac_{sp}"] = float(max_b / max(bags,1))

    # composite score (lower is better)
    score = 0.0
    score += 1000.0 * m["violations"]

    # emphasize val/test
    score += 3.0*m["jsd_label_train"] + 10.0*m["jsd_label_val"] + 10.0*m["jsd_label_test"]
    score += 2.0*m["ks_donorbag_train"] + 6.0*m["ks_donorbag_val"] + 6.0*m["ks_donorbag_test"]

    # donor domination penalty (val/test focus)
    score += 60.0*m["max_donor_frac_val"] + 60.0*m["max_donor_frac_test"]

    # penalty if hugging control minimum (instability risk)
    score += 0.8*max(0, 16 - m["controls_train"])  # train controls 15 is penalized
    score += 1.2*max(0, 6  - m["controls_val"])    # val controls 5 is penalized
    score += 1.2*max(0, 6  - m["controls_test"])   # test controls 5 is penalized

    # small penalty for deviating from 42/42 (val/test symmetry)
    score += 0.2*abs(m["bags_val"]-TARGET["val"]) + 0.2*abs(m["bags_test"]-TARGET["test"])

    m["score"] = float(score)
    rows.append(m)

rank_df = pd.DataFrame(rows).sort_values(["score","seed"]).reset_index(drop=True)
print(rank_df.head(10)[["seed","score",
                        "bags_val","bags_test","controls_train","controls_val","controls_test",
                        "donors_val","donors_test",
                        "jsd_label_val","jsd_label_test",
                        "ks_donorbag_val","ks_donorbag_test"]])

best = rank_df.iloc[0]
print("\nBEST SEED:", int(best["seed"]), "score:", best["score"])


   seed      score  bags_val  bags_test  controls_train  controls_val  \
0    11   9.335258        42         42              18             5   
1     5  11.718498        42         42              18             5   
2     2  12.326039        43         41              18             5   
3    12  12.534410        42         42              18             5   
4     6  13.398861        42         42              17             5   
5     0  13.698811        42         42              18             5   
6     4  13.740864        43         41              17             6   
7    19  15.714945        42         42              18             5   
8    16  15.785272        42         42              18             5   
9    14  15.830683        42         42              18             5   

   controls_test  donors_val  donors_test  jsd_label_val  jsd_label_test  \
0              5          36           33       0.000993        0.001532   
1              5          31           32   